In [35]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [36]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [37]:
Path(root / "csv_cleaned").mkdir(exist_ok=True)

Cargar archivos csv

In [38]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [45]:
def create_id(row):
    id_string = f"{row["asignatura"]}_{row["año"]}_{row["convocatoria"]}"
    if not pd.isna(row["ejercicio"]):
        row["ejercicio"] = row["ejercicio"].replace(",", "").replace(" ", "_")
        id_string += f"_{row["ejercicio"]}"
    return id_string.lower()

In [41]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción").replace(".", ",") if not pd.isna(x) else x)
    
    # create id
    df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
    # rearrange columns
    df = df[["id"] + list(df.columns[:-2])]

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto) if not pd.isna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto).strip() if not pd.isna(texto) else texto)

    
    
    # export clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)
    
    df.to_csv(output_dir / "enunciados_ejercicios_clean.csv", encoding="utf-8", index = False)

In [46]:
clean_dataframe(df)

In [47]:
df_cleaned = pd.read_csv(root / "csv_cleaned/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [48]:
df_cleaned.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
315,física_2022_reserva 3_ejercicio_b2,Física,Física,2022,Reserva 3,Ejercicio B2,Campo eléctrico y magnético,FISICA. 2022. RESERVA 3. EJERCICIO B2,a) Razone la veracidad o falsedad de la siguie...,2022 - Campo eléctrico y magnético.pdf
2786,mates ii_2022_reserva 3_ejercicio_5,Mates II,Mates II,2022,Reserva 3,Ejercicio 5,Sistemas De Ecuaciones Lineales,MATEMÁTICAS II. 2022. RESERVA 3. EJERCICIO 5,La suma de los seguidores en una determinada r...,2022 - Sistemas De Ecuaciones Lineales.pdf
2387,mates ii_2014_reserva 2_ejercicio_3_opción_b,Mates II,Mates II,2014,Reserva 2,"Ejercicio 3, opción B,",Matrices Y Determinantes,MATEMÁTICAS II. 2014. RESERVA 2. EJERCICIO 3. ...,1m 1  1 1\nConsidera las matrices A  ...,2014 - Matrices Y Determinantes.pdf


In [49]:
df_cleaned.ejercicio.unique()

<StringArray>
['Ejercicio 1, opción A', 'Ejercicio 1, opción B', 'Ejercicio 2, opción A',
 'Ejercicio 2, opción B', 'Ejercicio 3, opción A', 'Ejercicio 3, opción B',
 'Ejercicio 4, opción A', 'Ejercicio 4, opción B',           'Ejercicio 1',
           'Ejercicio 2',
 ...
         'Ejercicio C 2',         'Ejercicio C 3',         'Ejercicio C 4',
         'Ejercicio B 1',         'Ejercicio C 1',         'Ejercicio B 3',
         'Ejercicio B 2',         'Ejercicio B 6',          'Ejercicio 1B',
          'Ejercicio 2A']
Length: 218, dtype: str